# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
To reduce multicollinearity and improve balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


exp_name = "E8b_vascular_signal"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8b_vascular_signal_H5.csv (38341, 69)
label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,FOCLSIGN,HACHIN,HACHIN_GE_4,HACHIN_GE_7,STROKE_TIA_ANY,VASC_MED_COUNT,GAIT_ANY,NACCMOM,NACCDAD,SIVDFIND
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,0.0,0,NaN,NaN,NaN
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,2.0,0,1.0,1.0,0.0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,1.0,1,1.0,0.0,NaN
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [4]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        #X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [5]:
import numpy as np
import pandas as pd


df4 = df.copy()  

# Keeping only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Dropping IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns that are always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline", "NACCUDSD"]
                 if c in df.columns]

    # E8: engineered vascular burden; dropping raw components to avoid collinearity
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # E9: exec ratio; Dropping raw Trails
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # E10: building exec-memory axis (train-only z-scoring later), then Dropping raw
    # Here, what is dropped is only marked AFTER computing engineered features per split.

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Applying categorical cleanup before splitting
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using train stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Dropping raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14537
label
NoDementia_Hy    11083
AD                3013
PD_LBD             289
Vascular           152
Name: count, dtype: int64
Train: (11629, 62) Test: (2908, 62)
Train label counts:
 label
NoDementia_Hy    8866
AD               2410
PD_LBD            231
Vascular          122
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                603
PD_LBD             58
Vascular           30
Name: count, dtype: int64


In [6]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )

Preprocessing pipelines (sparse for LR/boosting)

In [7]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
print(sorted([c for c in X_train.columns]))



Num cols: 59 Cat cols: 3
['NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC', 'VASC_MED_COUNT']
['ANIMALS', 'APNEA', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CBSTROKE', 'CBTIA', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'CVAFIB', 'CVANGINA', 'CVANGIO', 'CVBYPASS', 'CVCHF', 'CVD', 'CVDCOG', 'CVDIMAG', 'CVHVALVE', 'CVOTHR', 'DEP2YRS', 'DIABETES', 'EDUC', 'FOCLSIGN', 'FOCLSYM', 'GAITDIS', 'GAIT_ANY', 'HACHIN', 'HACHIN_GE_4', 'HACHIN_GE_7', 'HISPANIC', 'HYPERCHO', 'HYPERTEN', 'IMAGEWMH', 'IMAGMWMH', 'NACCAGE', 'NACCDAD', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCMOM', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SIVDFIND', 'SLOWINGL', 'SLOWINGR', 'STROKE_TIA_ANY', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC', 'VASC_MED_COUNT']


Model A: Logistic Regression (final settings)

# Section 3C: Logistic Regression (balanced weights + vascular boost + threshold)

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline
# from sklearn.base import clone
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
# import numpy as np

# # 1) Systematic class weights (balanced) + optional vascular boost
# classes = np.unique(y_train)

# base_w = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=y_train
# )
# class_weight_dict = dict(zip(classes, base_w))

# VAS_MULTIPLIER = 1.1   # try: 1.0, 1.2, 1.3, 1.5
# class_weight_dict["Vascular"] = class_weight_dict.get("Vascular", 1.0) * VAS_MULTIPLIER

# print("Final class weights:", class_weight_dict)

# # 2) Fit multinomial logistic regression
# clf_lr = Pipeline(steps=[
#     ("preprocess", clone(preprocess)),
#     ("model", LogisticRegression(
#         solver="lbfgs",          # supports multinomial automatically
#         max_iter=8000,
#         class_weight=class_weight_dict,
#         C=0.1
#     ))
# ])

# clf_lr.fit(X_train, y_train)

# # 3) Default prediction
# pred_lr = clf_lr.predict(X_test)

# # 4) OPTIONAL: vascular threshold override (often boosts vascular recall)
# USE_VASC_THRESHOLD = False
# T_VAS = 0.18  # try: 0.15–0.25

# if USE_VASC_THRESHOLD:
#     probs = clf_lr.predict_proba(X_test)
#     order = clf_lr.named_steps["model"].classes_
#     vas_idx = list(order).index("Vascular")
#     p_vas = probs[:, vas_idx]

#     pred_lr = pred_lr.copy()
#     pred_lr[p_vas >= T_VAS] = "Vascular"

# print("\n=== Logistic Regression (balanced weights + vascular boost + optional threshold) ===")
# print(classification_report(y_test, pred_lr, digits=4, labels=labels))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))

# Section 3A: Logistic Regression (custom weights, C=0.1)

In [8]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6235    0.7114    0.6646       603
NoDementia_Hy     0.9250    0.8791    0.9015      2217
       PD_LBD     0.4512    0.6379    0.5286        58
     Vascular     0.3226    0.3333    0.3279        30

     accuracy                         0.8339      2908
    macro avg     0.5806    0.6405    0.6056      2908
 weighted avg     0.8468    0.8339    0.8390      2908

Balanced accuracy: 0.6404557690758564
[[ 429  142   20   12]
 [ 235 1949   25    8]
 [  14    6   37    1]
 [  10   10    0   10]]


# Section 3G: Tuned Logistic Regression

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

# 1) Created train/validation split from TRAIN only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 2) Fitted LR on inner-train

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_tr, y_tr)

# 3) Tuned vascular threshold on validation set
proba_val = clf_lr.predict_proba(X_val)
class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

thresholds = np.linspace(0.05, 0.50, 20)
results = []

for T in thresholds:
    preds = []
    for p in proba_val:
        if p[vas_idx] >= T:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    ba = balanced_accuracy_score(y_val, preds)

    # vascular recall
    yv = np.array(y_val)
    pv = np.array(preds)
    denom = np.sum(yv == "Vascular")
    vrec = (np.sum((yv == "Vascular") & (pv == "Vascular")) / denom) if denom else 0.0

    results.append((T, ba, vrec))

best_T, best_ba, best_vrec = max(results, key=lambda x: x[1])
print(f"Best T_VAS (by val BA): {best_T:.3f} | val BA={best_ba:.4f} | val VascRecall={best_vrec:.4f}")

# 4) Retrained LR on FULL TRAIN and evaluate once on TEST using tuned threshold
clf_lr.fit(X_train, y_train)

class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

proba_test = clf_lr.predict_proba(X_test)
preds_test = []

for p in proba_test:
    if p[vas_idx] >= best_T:
        preds_test.append("Vascular")
    else:
        preds_test.append(class_order[np.argmax(p)])

print("\n=== Logistic Regression (Vascular threshold tuned) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, preds_test))
print(confusion_matrix(y_test, preds_test, labels=labels))

Best T_VAS (by val BA): 0.074 | val BA=0.6517 | val VascRecall=0.6000

=== Logistic Regression (Vascular threshold tuned) ===
               precision    recall  f1-score   support

           AD     0.6358    0.5473    0.5882       603
NoDementia_Hy     0.9331    0.8489    0.8890      2217
       PD_LBD     0.4861    0.6034    0.5385        58
     Vascular     0.0700    0.7000    0.1273        30

     accuracy                         0.7799      2908
    macro avg     0.5313    0.6749    0.5357      2908
 weighted avg     0.8536    0.7799    0.8118      2908

Balanced accuracy: 0.6749017151190526
[[ 330  124   17  132]
 [ 173 1882   20  142]
 [  13    5   35    5]
 [   3    6    0   21]]


In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     classification_report,
#     confusion_matrix,
#     balanced_accuracy_score,
#     f1_score,
#     precision_recall_fscore_support
# )
# from sklearn.pipeline import Pipeline
# import numpy as np
# import pandas as pd

# # --------------------------------------------------
# # 0) Labels / weights
# # --------------------------------------------------
# labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# # Slightly reduced vascular weight to help precision/F1
# custom_w = {
#     "NoDementia_Hy": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 5.0,
#     "Vascular": 6.0
# }

# # --------------------------------------------------
# # 1) Train / validation split from TRAIN only
# # --------------------------------------------------
# X_tr, X_val, y_tr, y_val = train_test_split(
#     X_train, y_train,
#     test_size=0.2,
#     random_state=42,
#     stratify=y_train
# )

# # --------------------------------------------------
# # 2) Candidate model settings
# #    One model change included: Elastic Net logistic regression
# # --------------------------------------------------
# model_settings = [
#     {
#         "name": "LR_L2_C0.1",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.1,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_L2_C0.05",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.05,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_ElasticNet_C0.1_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.1,
#         "l1_ratio": 0.2,
#     },
#     {
#         "name": "LR_ElasticNet_C0.05_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.05,
#         "l1_ratio": 0.2,
#     },
# ]

# thresholds = np.linspace(0.05, 0.40, 36)

# # --------------------------------------------------
# # Helper: apply vascular threshold override
# # --------------------------------------------------
# def predict_with_vascular_threshold(proba, class_order, threshold):
#     vas_idx = list(class_order).index("Vascular")
#     preds = []
#     for p in proba:
#         if p[vas_idx] >= threshold:
#             preds.append("Vascular")
#         else:
#             preds.append(class_order[np.argmax(p)])
#     return np.array(preds)

# # --------------------------------------------------
# # 3) Validation search:
# #    choose model + threshold by vascular F1
# #    tie-breakers: macro F1, balanced accuracy
# # --------------------------------------------------
# search_rows = []

# for cfg in model_settings:
#     clf = Pipeline(steps=[
#         ("preprocess", preprocess),
#         ("model", LogisticRegression(
#             max_iter=12000,
#             class_weight=custom_w,
#             C=cfg["C"],
#             penalty=cfg["penalty"],
#             solver=cfg["solver"],
#             l1_ratio=cfg["l1_ratio"],
#             n_jobs=None
#         ))
#     ])

#     clf.fit(X_tr, y_tr)

#     proba_val = clf.predict_proba(X_val)
#     class_order = clf.named_steps["model"].classes_

#     for T in thresholds:
#         preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

#         # overall metrics
#         ba = balanced_accuracy_score(y_val, preds_val)
#         macro_f1 = f1_score(y_val, preds_val, average="macro")

#         # per-class metrics
#         p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#             y_val, preds_val, labels=labels, zero_division=0
#         )

#         metric_map = {
#             lab: {"precision": p, "recall": r, "f1": f, "support": s}
#             for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
#         }

#         vprec = metric_map["Vascular"]["precision"]
#         vrec  = metric_map["Vascular"]["recall"]
#         vf1   = metric_map["Vascular"]["f1"]

#         search_rows.append({
#             "model_name": cfg["name"],
#             "penalty": cfg["penalty"],
#             "solver": cfg["solver"],
#             "C": cfg["C"],
#             "l1_ratio": cfg["l1_ratio"],
#             "threshold": T,
#             "val_bal_acc": ba,
#             "val_macro_f1": macro_f1,
#             "val_vasc_precision": vprec,
#             "val_vasc_recall": vrec,
#             "val_vasc_f1": vf1,
#         })

# search_df = pd.DataFrame(search_rows)

# # Sort by vascular F1 first, then macro F1, then BA
# search_df = search_df.sort_values(
#     ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
#     ascending=[False, False, False]
# ).reset_index(drop=True)

# print("Top validation settings:")
# display(search_df.head(10))

# best_row = search_df.iloc[0]
# best_model_name = best_row["model_name"]
# best_T = float(best_row["threshold"])

# print(
#     f"Best setting: {best_model_name} | "
#     f"T_VAS={best_T:.3f} | "
#     f"val VascF1={best_row['val_vasc_f1']:.4f} | "
#     f"val MacroF1={best_row['val_macro_f1']:.4f} | "
#     f"val BA={best_row['val_bal_acc']:.4f} | "
#     f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
#     f"val VascRec={best_row['val_vasc_recall']:.4f}"
# )

# # --------------------------------------------------
# # 4) Refit best model on FULL TRAIN
# # --------------------------------------------------
# best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

# clf_best = Pipeline(steps=[
#     ("preprocess", preprocess),
#     ("model", LogisticRegression(
#         max_iter=12000,
#         class_weight=custom_w,
#         C=best_cfg["C"],
#         penalty=best_cfg["penalty"],
#         solver=best_cfg["solver"],
#         l1_ratio=best_cfg["l1_ratio"],
#         n_jobs=None
#     ))
# ])

# clf_best.fit(X_train, y_train)

# class_order = clf_best.named_steps["model"].classes_
# proba_test = clf_best.predict_proba(X_test)
# preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# # --------------------------------------------------
# # 5) Final test evaluation
# # --------------------------------------------------
# print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
# print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

# test_bal_acc = balanced_accuracy_score(y_test, preds_test)
# test_macro_f1 = f1_score(y_test, preds_test, average="macro")
# test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
# test_cm = confusion_matrix(y_test, preds_test, labels=labels)

# print("Balanced accuracy:", test_bal_acc)
# print("Macro F1:", test_macro_f1)
# print("Weighted F1:", test_weighted_f1)
# print(confusion_matrix(y_test, preds_test, labels=labels))

# # Optional: print vascular-specific test metrics cleanly
# p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#     y_test, preds_test, labels=labels, zero_division=0
# )
# test_metric_map = {
#     lab: {"precision": p, "recall": r, "f1": f, "support": s}
#     for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
# }

# print("\nTest vascular metrics:")
# print(test_metric_map["Vascular"])

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import (
#     classification_report,
#     confusion_matrix,
#     balanced_accuracy_score,
#     f1_score,
#     precision_recall_fscore_support
# )
# from sklearn.pipeline import Pipeline
# import numpy as np
# import pandas as pd

# # --------------------------------------------------
# # 0) Labels / weights
# # --------------------------------------------------
# labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# custom_w = {
#     "NoDementia_Hy": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 5.0,
#     "Vascular": 6.0
# }

# # --------------------------------------------------
# # 1) Candidate model settings
# # --------------------------------------------------
# model_settings = [
#     {
#         "name": "LR_L2_C0.1",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.1,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_L2_C0.05",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.05,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_ElasticNet_C0.1_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.1,
#         "l1_ratio": 0.2,
#     },
#     {
#         "name": "LR_ElasticNet_C0.05_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.05,
#         "l1_ratio": 0.2,
#     },
# ]

# thresholds = np.linspace(0.05, 0.40, 36)

# # --------------------------------------------------
# # Helper: apply vascular threshold override
# # --------------------------------------------------
# def predict_with_vascular_threshold(proba, class_order, threshold):
#     vas_idx = list(class_order).index("Vascular")
#     preds = []
#     for p in proba:
#         if p[vas_idx] >= threshold:
#             preds.append("Vascular")
#         else:
#             preds.append(class_order[np.argmax(p)])
#     return np.array(preds)

# # --------------------------------------------------
# # 2) 3-fold CV search (FAST VERSION)
# # --------------------------------------------------
# skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# cv_rows = []

# for cfg in model_settings:
#     print(f"Running model config: {cfg['name']}")
    
#     # Store metrics by threshold across folds
#     threshold_metric_store = {
#         T: {
#             "bal_acc": [],
#             "macro_f1": [],
#             "vasc_precision": [],
#             "vasc_recall": [],
#             "vasc_f1": []
#         }
#         for T in thresholds
#     }

#     for fold_id, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
#         print(f"  Fold {fold_id}/3")

#         X_tr_fold = X_train.iloc[tr_idx]
#         X_val_fold = X_train.iloc[val_idx]
#         y_tr_fold = y_train.iloc[tr_idx]
#         y_val_fold = y_train.iloc[val_idx]

#         clf = Pipeline(steps=[
#             ("preprocess", preprocess),
#             ("model", LogisticRegression(
#                 max_iter=12000,
#                 class_weight=custom_w,
#                 C=cfg["C"],
#                 penalty=cfg["penalty"],
#                 solver=cfg["solver"],
#                 l1_ratio=cfg["l1_ratio"],
#                 n_jobs=None
#             ))
#         ])

#         # FIT ONCE PER FOLD
#         clf.fit(X_tr_fold, y_tr_fold)

#         # PREDICT PROBA ONCE PER FOLD
#         proba_val = clf.predict_proba(X_val_fold)
#         class_order = clf.named_steps["model"].classes_

#         # LOOP OVER THRESHOLDS USING SAME PROBAS
#         for T in thresholds:
#             preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

#             ba = balanced_accuracy_score(y_val_fold, preds_val)
#             macro_f1 = f1_score(y_val_fold, preds_val, average="macro")

#             p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#                 y_val_fold, preds_val, labels=labels, zero_division=0
#             )

#             metric_map = {
#                 lab: {"precision": p, "recall": r, "f1": f, "support": s}
#                 for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
#             }

#             threshold_metric_store[T]["bal_acc"].append(ba)
#             threshold_metric_store[T]["macro_f1"].append(macro_f1)
#             threshold_metric_store[T]["vasc_precision"].append(metric_map["Vascular"]["precision"])
#             threshold_metric_store[T]["vasc_recall"].append(metric_map["Vascular"]["recall"])
#             threshold_metric_store[T]["vasc_f1"].append(metric_map["Vascular"]["f1"])

#     # Aggregate across folds for this model config
#     for T in thresholds:
#         cv_rows.append({
#             "model_name": cfg["name"],
#             "penalty": cfg["penalty"],
#             "solver": cfg["solver"],
#             "C": cfg["C"],
#             "l1_ratio": cfg["l1_ratio"],
#             "threshold": T,
#             "cv_bal_acc_mean": np.mean(threshold_metric_store[T]["bal_acc"]),
#             "cv_bal_acc_std": np.std(threshold_metric_store[T]["bal_acc"]),
#             "cv_macro_f1_mean": np.mean(threshold_metric_store[T]["macro_f1"]),
#             "cv_macro_f1_std": np.std(threshold_metric_store[T]["macro_f1"]),
#             "cv_vasc_precision_mean": np.mean(threshold_metric_store[T]["vasc_precision"]),
#             "cv_vasc_recall_mean": np.mean(threshold_metric_store[T]["vasc_recall"]),
#             "cv_vasc_f1_mean": np.mean(threshold_metric_store[T]["vasc_f1"]),
#             "cv_vasc_f1_std": np.std(threshold_metric_store[T]["vasc_f1"]),
#         })

# cv_df = pd.DataFrame(cv_rows)

# # Sort by mean vascular F1 first, then mean macro F1, then mean BA
# cv_df = cv_df.sort_values(
#     ["cv_vasc_f1_mean", "cv_macro_f1_mean", "cv_bal_acc_mean"],
#     ascending=[False, False, False]
# ).reset_index(drop=True)

# print("\nTop 15 CV settings:")
# display(cv_df.head(15))

# best_row = cv_df.iloc[0]
# best_model_name = best_row["model_name"]
# best_T = float(best_row["threshold"])

# print(
#     f"Best CV setting: {best_model_name} | "
#     f"T_VAS={best_T:.3f} | "
#     f"CV mean VascF1={best_row['cv_vasc_f1_mean']:.4f} ± {best_row['cv_vasc_f1_std']:.4f} | "
#     f"CV mean MacroF1={best_row['cv_macro_f1_mean']:.4f} ± {best_row['cv_macro_f1_std']:.4f} | "
#     f"CV mean BA={best_row['cv_bal_acc_mean']:.4f} ± {best_row['cv_bal_acc_std']:.4f}"
# )

# # --------------------------------------------------
# # 3) Refit best model on FULL TRAIN
# # --------------------------------------------------
# best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

# clf_best = Pipeline(steps=[
#     ("preprocess", preprocess),
#     ("model", LogisticRegression(
#         max_iter=12000,
#         class_weight=custom_w,
#         C=best_cfg["C"],
#         penalty=best_cfg["penalty"],
#         solver=best_cfg["solver"],
#         l1_ratio=best_cfg["l1_ratio"],
#         n_jobs=None
#     ))
# ])

# clf_best.fit(X_train, y_train)

# class_order = clf_best.named_steps["model"].classes_
# proba_test = clf_best.predict_proba(X_test)
# preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# # --------------------------------------------------
# # 4) Final test evaluation
# # --------------------------------------------------
# print("\n=== Logistic Regression (3-fold CV selected model + vascular threshold) ===")
# print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

# test_bal_acc = balanced_accuracy_score(y_test, preds_test)
# test_macro_f1 = f1_score(y_test, preds_test, average="macro")
# test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")

# print("Balanced accuracy:", test_bal_acc)
# print("Macro F1:", test_macro_f1)
# print("Weighted F1:", test_weighted_f1)
# print(confusion_matrix(y_test, preds_test, labels=labels))

# p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#     y_test, preds_test, labels=labels, zero_division=0
# )
# test_metric_map = {
#     lab: {"precision": p, "recall": r, "f1": f, "support": s}
#     for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
# }

# print("\nTest vascular metrics:")
# print(test_metric_map["Vascular"])

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# --------------------------------------------------
# 0) Labels / weights
# --------------------------------------------------
labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# Slightly reduced vascular weight to help precision/F1
custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0
}

# --------------------------------------------------
# 1) Train / validation split from TRAIN only
# --------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# --------------------------------------------------
# 2) Candidate model settings
#    One model change included: Elastic Net logistic regression
# --------------------------------------------------
model_settings = [
    {
        "name": "LR_L2_C0.1",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.1,
        "l1_ratio": None,
    },
    {
        "name": "LR_L2_C0.05",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.05,
        "l1_ratio": None,
    },
    {
        "name": "LR_ElasticNet_C0.1_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0.2,
    },
    {
        "name": "LR_ElasticNet_C0.05_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0.2,
    },
]

thresholds = np.linspace(0.05, 0.40, 36)

# --------------------------------------------------
# Helper: apply vascular threshold override
# --------------------------------------------------
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

# --------------------------------------------------
# 3) Validation search:
#    choose model + threshold by vascular F1
#    tie-breakers: macro F1, balanced accuracy
# --------------------------------------------------
search_rows = []

for cfg in model_settings:
    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=12000,
            class_weight=custom_w,
            C=cfg["C"],
            penalty=cfg["penalty"],
            solver=cfg["solver"],
            l1_ratio=cfg["l1_ratio"],
            n_jobs=None
        ))
    ])

    clf.fit(X_tr, y_tr)

    proba_val = clf.predict_proba(X_val)
    class_order = clf.named_steps["model"].classes_

    for T in thresholds:
        preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

        # overall metrics
        ba = balanced_accuracy_score(y_val, preds_val)
        macro_f1 = f1_score(y_val, preds_val, average="macro")

        # per-class metrics
        p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
            y_val, preds_val, labels=labels, zero_division=0
        )

        metric_map = {
            lab: {"precision": p, "recall": r, "f1": f, "support": s}
            for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
        }

        vprec = metric_map["Vascular"]["precision"]
        vrec  = metric_map["Vascular"]["recall"]
        vf1   = metric_map["Vascular"]["f1"]

        search_rows.append({
            "model_name": cfg["name"],
            "penalty": cfg["penalty"],
            "solver": cfg["solver"],
            "C": cfg["C"],
            "l1_ratio": cfg["l1_ratio"],
            "threshold": T,
            "val_bal_acc": ba,
            "val_macro_f1": macro_f1,
            "val_vasc_precision": vprec,
            "val_vasc_recall": vrec,
            "val_vasc_f1": vf1,
        })

search_df = pd.DataFrame(search_rows)

# Sort by vascular F1 first, then macro F1, then BA
search_df = search_df.sort_values(
    ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Top validation settings:")
display(search_df.head(10))

best_row = search_df.iloc[0]
best_model_name = best_row["model_name"]
best_T = float(best_row["threshold"])

print(
    f"Best setting: {best_model_name} | "
    f"T_VAS={best_T:.3f} | "
    f"val VascF1={best_row['val_vasc_f1']:.4f} | "
    f"val MacroF1={best_row['val_macro_f1']:.4f} | "
    f"val BA={best_row['val_bal_acc']:.4f} | "
    f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
    f"val VascRec={best_row['val_vasc_recall']:.4f}"
)

# --------------------------------------------------
# 4) Refit best model on FULL TRAIN
# --------------------------------------------------
best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        class_weight=custom_w,
        C=best_cfg["C"],
        penalty=best_cfg["penalty"],
        solver=best_cfg["solver"],
        l1_ratio=best_cfg["l1_ratio"],
        n_jobs=None
    ))
])

clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# --------------------------------------------------
# 5) Final test evaluation
# --------------------------------------------------
print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
test_cm = confusion_matrix(y_test, preds_test, labels=labels)

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

# Optional: print vascular-specific test metrics cleanly
p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packag

Top validation settings:


,model_name,penalty,solver,C,l1_ratio,threshold,val_bal_acc,val_macro_f1,val_vasc_precision,val_vasc_recall,val_vasc_f1
0,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.33,0.641963,0.610168,0.209302,0.36,0.264706
1,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.32,0.641822,0.609131,0.204545,0.36,0.260870
2,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.36,0.635869,0.606174,0.204545,0.36,0.260870
3,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.30,0.641304,0.607867,0.200000,0.36,0.257143
4,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.31,0.641304,0.607867,0.200000,0.36,0.257143
5,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.35,0.635869,0.605414,0.200000,0.36,0.257143
6,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.29,0.641304,0.607133,0.195652,0.36,0.253521
7,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.33,0.635728,0.604430,0.195652,0.36,0.253521
8,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.34,0.635728,0.604430,0.195652,0.36,0.253521
9,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.27,0.640785,0.605920,0.191489,0.36,0.250000


Best setting: LR_ElasticNet_C0.1_l1r0.2 | T_VAS=0.330 | val VascF1=0.2647 | val MacroF1=0.6102 | val BA=0.6420 | val VascPrec=0.2093 | val VascRec=0.3600


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===
               precision    recall  f1-score   support

           AD     0.6235    0.7032    0.6610       603
NoDementia_Hy     0.9238    0.8805    0.9016      2217
       PD_LBD     0.4805    0.6379    0.5481        58
     Vascular     0.2895    0.3667    0.3235        30

     accuracy                         0.8336      2908
    macro avg     0.5793    0.6471    0.6086      2908
 weighted avg     0.8462    0.8336    0.8387      2908

Balanced accuracy: 0.6470544289115449
Macro F1: 0.6085612711062712
Weighted F1: 0.838698737999208
[[ 424  146   18   15]
 [ 233 1952   22   10]
 [  14    5   37    2]
 [   9   10    0   11]]

Test vascular metrics:
{'precision': np.float64(0.2894736842105263), 'recall': np.float64(0.36666666666666664), 'f1': np.float64(0.3235294117647059), 'support': np.int64(30)}


c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



## Clean post-best-model section

Everything above this point is preserved. The sections below replace the earlier mixed-label / duplicate-function blocks with a clean, string-label-only workflow.

Recommended order:
1. Shared helpers
2. Logistic regression + risk-stratified vascular rule
3. CatBoost multiclass
4. Hierarchical CatBoost


In [11]:

# Shared helpers (string-label workflow)
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]


def eval_predictions(y_true, y_pred, labels=labels, title="Model"):
    print(f"=== {title} ===")
    print(classification_report(y_true, y_pred, digits=4, labels=labels, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
    print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(cm)

    p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    return pd.DataFrame({
        "class": labels,
        "precision": p_arr,
        "recall": r_arr,
        "f1": f_arr,
        "support": s_arr,
    })


def add_vascular_features(df):
    df = df.copy()

    # Use whichever variables are present in the cohort file.
    candidate_cols = [
        "CBSTROKE", "CBTIA", "CVAFIB", "DIABETES", "HYPERTEN", "HYPERCHO",
        "CVCHF", "CVANGINA", "CVBYPASS", "CVHVALVE",
        "NACCAC", "NACCLIPL", "NACCBETA", "NACCCCBS", "NACCDIUR", "NACCANGI", "NACCVASD",
        "DIABET", "HYPERT", "AFIBRILL", "SLEEPAP"
    ]
    for c in candidate_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    burden_components = [
        c for c in ["CBSTROKE", "CBTIA", "CVAFIB", "DIABETES", "HYPERTEN", "HYPERCHO", "CVCHF", "CVANGINA"]
        if c in df.columns
    ]
    if burden_components:
        df["vascular_burden_score"] = df[burden_components].fillna(0).sum(axis=1)

    if "CBSTROKE" in df.columns and "HYPERTEN" in df.columns:
        df["stroke_x_htn"] = df["CBSTROKE"].fillna(0) * df["HYPERTEN"].fillna(0)
    if "DIABETES" in df.columns and "HYPERTEN" in df.columns:
        df["dm_x_htn"] = df["DIABETES"].fillna(0) * df["HYPERTEN"].fillna(0)

    return df


def compute_vascular_risk_stratum(df):
    df = add_vascular_features(df)
    if "vascular_burden_score" not in df.columns:
        # fall back to a single low-risk stratum if burden score cannot be built
        return pd.Series(["low"] * len(df), index=df.index)

    score = df["vascular_burden_score"].fillna(0)
    strata = pd.Series(index=df.index, dtype="object")
    strata[score <= 1] = "low"
    strata[(score >= 2) & (score <= 3)] = "medium"
    strata[score >= 4] = "high"
    return strata


def fit_stratum_specific_thresholds(y_val, val_proba, val_df, target_class="Vascular"):
    strata = compute_vascular_risk_stratum(val_df)
    thresholds = {}
    rows = []
    target_idx = labels.index(target_class)

    for s in ["low", "medium", "high"]:
        mask = (strata == s).values
        if mask.sum() == 0:
            continue

        y_sub = np.asarray(y_val)[mask]
        proba_sub = val_proba[mask]
        if np.sum(y_sub == target_class) == 0:
            continue

        best = None
        for thr in np.linspace(0.05, 0.70, 66):
            pred = np.array([labels[np.argmax(p)] for p in proba_sub], dtype=object)
            pred[proba_sub[:, target_idx] >= thr] = target_class

            p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
                y_sub, pred, labels=[target_class], zero_division=0
            )
            row = {
                "stratum": s,
                "threshold": float(thr),
                "precision": float(p_arr[0]),
                "recall": float(r_arr[0]),
                "f1": float(f_arr[0]),
                "support": int(s_arr[0]),
            }
            if best is None or row["f1"] > best["f1"]:
                best = row

        thresholds[s] = best["threshold"]
        rows.append(best)

    return thresholds, pd.DataFrame(rows)


def predict_with_risk_stratified_margin_rule(proba, X_df, stratum_thresholds, margin=0.08, target_class="Vascular"):
    pred = np.array([labels[np.argmax(p)] for p in proba], dtype=object)
    target_idx = labels.index(target_class)
    strata = compute_vascular_risk_stratum(X_df)

    top_prob = proba.max(axis=1)
    for s in ["low", "medium", "high"]:
        if s not in stratum_thresholds:
            continue
        thr = stratum_thresholds[s]
        mask = (strata == s).values
        override = (
            mask
            & (proba[:, target_idx] >= thr)
            & ((top_prob - proba[:, target_idx]) <= margin)
        )
        pred[override] = target_class
    return pred


## Logistic regression + risk-stratified vascular rule

In [12]:

# Use the already-fitted best logistic model from the preserved section above: clf_best
X_val_fe = add_vascular_features(X_val)
X_test_fe = add_vascular_features(X_test)

val_proba_lr = clf_best.predict_proba(X_val)
stratum_thresholds_lr, table_lr = fit_stratum_specific_thresholds(
    y_val, val_proba_lr, X_val_fe, target_class="Vascular"
)
print("Stratum-specific vascular thresholds:", stratum_thresholds_lr)
display(table_lr)

test_proba_lr = clf_best.predict_proba(X_test)
test_pred_lr_strat = predict_with_risk_stratified_margin_rule(
    test_proba_lr,
    X_test_fe,
    stratum_thresholds_lr,
    margin=0.08,
    target_class="Vascular",
)

lr_strat_results = eval_predictions(
    y_test,
    test_pred_lr_strat,
    title="Current logistic model + risk-stratified vascular rule",
)
display(lr_strat_results)


Stratum-specific vascular thresholds: {'low': 0.09, 'medium': 0.3, 'high': 0.37999999999999995}


,stratum,threshold,precision,recall,f1,support
0,low,0.09,0.125000,0.500000,0.200000,6
1,medium,0.30,0.222222,0.461538,0.300000,13
2,high,0.38,0.277778,0.833333,0.416667,6


=== Current logistic model + risk-stratified vascular rule ===
               precision    recall  f1-score   support

           AD     0.6213    0.7048    0.6605       603
NoDementia_Hy     0.9238    0.8809    0.9019      2217
       PD_LBD     0.4805    0.6379    0.5481        58
     Vascular     0.2727    0.3000    0.2857        30

     accuracy                         0.8336      2908
    macro avg     0.5746    0.6309    0.5990      2908
 weighted avg     0.8456    0.8336    0.8384      2908

Balanced accuracy: 0.6309151209407986
Macro F1: 0.5990458330333648
Weighted F1: 0.8383982416311959
[[ 425  146   18   14]
 [ 234 1953   22    8]
 [  14    5   37    2]
 [  11   10    0    9]]


,class,precision,recall,f1,support
0,AD,0.621345,0.704809,0.660451,603
1,NoDementia_Hy,0.923841,0.880920,0.901870,2217
2,PD_LBD,0.480519,0.637931,0.548148,58
3,Vascular,0.272727,0.300000,0.285714,30


## CatBoost multiclass model

In [13]:

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    random_seed=42,
    verbose=200,
    class_weights=[2.0, 1.0, 5.0, 6.0],  # AD, NoDementia_Hy, PD_LBD, Vascular in alphabetical label order? no, use auto from training labels below
)

# Safer: provide weights in the order of catboost's observed class_names
train_classes = sorted(pd.Series(y_train).unique())
class_weight_map = {
    "AD": 2.0,
    "NoDementia_Hy": 1.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0,
}
cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    random_seed=42,
    verbose=200,
    class_weights=[class_weight_map[c] for c in train_classes],
)

cat_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

val_proba_cat = cat_model.predict_proba(X_val)
cat_class_order = list(cat_model.classes_)
vas_idx = cat_class_order.index("Vascular")

best_cat = None
for thr in np.linspace(0.05, 0.60, 56):
    preds = np.array([cat_class_order[np.argmax(p)] for p in val_proba_cat], dtype=object)
    preds[val_proba_cat[:, vas_idx] >= thr] = "Vascular"

    p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
        y_val, preds, labels=["Vascular"], zero_division=0
    )
    row = {
        "threshold": float(thr),
        "precision": float(p_arr[0]),
        "recall": float(r_arr[0]),
        "f1": float(f_arr[0]),
        "support": int(s_arr[0]),
    }
    if best_cat is None or row["f1"] > best_cat["f1"]:
        best_cat = row

print("Best CatBoost vascular threshold on validation:", best_cat)

test_proba_cat = cat_model.predict_proba(X_test)
test_pred_cat = np.array([cat_class_order[np.argmax(p)] for p in test_proba_cat], dtype=object)
test_pred_cat[test_proba_cat[:, vas_idx] >= best_cat["threshold"]] = "Vascular"

cat_results = eval_predictions(
    y_test,
    test_pred_cat,
    title="CatBoost + tuned vascular threshold",
)
display(cat_results)


0:	learn: 0.7218189	test: 0.7187713	best: 0.7187713 (0)	total: 141ms	remaining: 4m 41s
200:	learn: 0.8019819	test: 0.8032604	best: 0.8043328 (192)	total: 1.43s	remaining: 12.8s
400:	learn: 0.8274106	test: 0.8175216	best: 0.8178122 (394)	total: 2.56s	remaining: 10.2s
600:	learn: 0.8458328	test: 0.8379279	best: 0.8379279 (598)	total: 3.75s	remaining: 8.73s
800:	learn: 0.8603185	test: 0.8575423	best: 0.8575423 (800)	total: 5.04s	remaining: 7.55s
1000:	learn: 0.8746193	test: 0.8721205	best: 0.8734614 (963)	total: 8.58s	remaining: 8.56s
1200:	learn: 0.8842765	test: 0.8837808	best: 0.8843923 (1184)	total: 12.3s	remaining: 8.16s
1400:	learn: 0.8967115	test: 0.8921165	best: 0.8924236 (1336)	total: 16.5s	remaining: 7.05s
1600:	learn: 0.9029142	test: 0.9005296	best: 0.9005296 (1565)	total: 20.4s	remaining: 5.09s
1800:	learn: 0.9113879	test: 0.9081517	best: 0.9081517 (1774)	total: 24.1s	remaining: 2.67s
1999:	learn: 0.9190326	test: 0.9160603	best: 0.9161064 (1970)	total: 27.5s	remaining: 0us

bes

,class,precision,recall,f1,support
0,AD,0.672107,0.751244,0.709475,603
1,NoDementia_Hy,0.926335,0.907533,0.916838,2217
2,PD_LBD,0.574074,0.534483,0.553571,58
3,Vascular,0.375000,0.100000,0.157895,30


## Hierarchical CatBoost (NoDementia vs Dementia, then subtype)

In [14]:

from catboost import CatBoostClassifier

NO_DEM = "NoDementia_Hy"

# Stage 1: dementia vs no dementia
# 0 = No dementia, 1 = Dementia subtype
y_train_s1 = (pd.Series(y_train).values != NO_DEM).astype(int)
y_val_s1   = (pd.Series(y_val).values != NO_DEM).astype(int)
y_test_s1  = (pd.Series(y_test).values != NO_DEM).astype(int)

print("Stage 1 train distribution:", np.unique(y_train_s1, return_counts=True))
print("Stage 1 val distribution:", np.unique(y_val_s1, return_counts=True))

stage1 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=200,
    class_weights=[1.0, 1.5],
)

stage1.fit(
    X_train,
    y_train_s1,
    eval_set=(X_val, y_val_s1),
    use_best_model=True,
)

# Stage 2: dementia subtype only
DEM_CLASS_TO_LOCAL = {"AD": 0, "PD_LBD": 1, "Vascular": 2}
LOCAL_TO_GLOBAL = {0: "AD", 1: "PD_LBD", 2: "Vascular"}

train_mask_s2 = pd.Series(y_train).values != NO_DEM
val_mask_s2   = pd.Series(y_val).values != NO_DEM

y_train_arr = pd.Series(y_train).values
y_val_arr = pd.Series(y_val).values

X_train_s2 = X_train.loc[train_mask_s2] if hasattr(X_train, "loc") else X_train[train_mask_s2]
X_val_s2   = X_val.loc[val_mask_s2] if hasattr(X_val, "loc") else X_val[val_mask_s2]

y_train_s2 = np.array([DEM_CLASS_TO_LOCAL[v] for v in y_train_arr[train_mask_s2]])
y_val_s2   = np.array([DEM_CLASS_TO_LOCAL[v] for v in y_val_arr[val_mask_s2]])

stage2 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=200,
    class_weights=[1.0, 2.0, 4.0],
)

stage2.fit(
    X_train_s2,
    y_train_s2,
    eval_set=(X_val_s2, y_val_s2),
    use_best_model=True,
)


Stage 1 train distribution: (array([0, 1]), array([8866, 2763]))
Stage 1 val distribution: (array([0, 1]), array([1773,  553]))
0:	learn: 0.7650304	test: 0.7651822	best: 0.7651822 (0)	total: 10.1ms	remaining: 15.2s
200:	learn: 0.8107417	test: 0.7977422	best: 0.7989997 (181)	total: 3.98s	remaining: 25.7s
400:	learn: 0.8357825	test: 0.8266751	best: 0.8266751 (400)	total: 7.53s	remaining: 20.6s
600:	learn: 0.8582474	test: 0.8461538	best: 0.8461538 (600)	total: 12.6s	remaining: 18.8s
800:	learn: 0.8772980	test: 0.8675373	best: 0.8701863 (793)	total: 16.2s	remaining: 14.1s
1000:	learn: 0.8929552	test: 0.8898542	best: 0.8908865 (997)	total: 19.2s	remaining: 9.57s
1200:	learn: 0.9056327	test: 0.9053089	best: 0.9053089 (1199)	total: 22.1s	remaining: 5.49s
1400:	learn: 0.9164022	test: 0.9161012	best: 0.9161012 (1394)	total: 24.9s	remaining: 1.76s
1499:	learn: 0.9194701	test: 0.9199382	best: 0.9209389 (1491)	total: 26.4s	remaining: 0us

bestTest = 0.9209388511
bestIteration = 1491

Shrink model 

CatBoostClassifier(class_weights=[1.0, 2.0, 4.0], depth=6, eval_metric='TotalF1', iterations=1500, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=200)

### Tune hierarchical thresholds on the validation set

In [15]:

# Stage 1 threshold
val_proba_s1 = stage1.predict_proba(X_val)[:, 1]
best_s1 = None
for thr in np.linspace(0.20, 0.80, 61):
    pred = (val_proba_s1 >= thr).astype(int)
    score = f1_score(y_val_s1, pred)
    row = {"threshold": float(thr), "f1": float(score)}
    if best_s1 is None or row["f1"] > best_s1["f1"]:
        best_s1 = row
print("Best Stage 1 threshold:", best_s1)

# Stage 2 vascular threshold among dementia cases
val_proba_s2 = stage2.predict_proba(X_val_s2)
vasc_local = DEM_CLASS_TO_LOCAL["Vascular"]
best_s2_vasc = None
for thr in np.linspace(0.05, 0.70, 66):
    pred_local = np.argmax(val_proba_s2, axis=1)
    pred_local[val_proba_s2[:, vasc_local] >= thr] = vasc_local

    pred_global = np.array([LOCAL_TO_GLOBAL[v] for v in pred_local], dtype=object)
    truth_global = np.array([LOCAL_TO_GLOBAL[v] for v in y_val_s2], dtype=object)

    p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
        truth_global, pred_global, labels=["Vascular"], zero_division=0
    )
    row = {
        "threshold": float(thr),
        "precision": float(p_arr[0]),
        "recall": float(r_arr[0]),
        "f1": float(f_arr[0]),
        "support": int(s_arr[0]),
    }
    if best_s2_vasc is None or row["f1"] > best_s2_vasc["f1"]:
        best_s2_vasc = row
print("Best Stage 2 vascular threshold:", best_s2_vasc)


Best Stage 1 threshold: {'threshold': 0.4700000000000001, 'f1': 0.9102333931777379}
Best Stage 2 vascular threshold: {'threshold': 0.38999999999999996, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 25}


### Hierarchical CatBoost test-time prediction

In [16]:

def hierarchical_predict(X):
    s1_proba = stage1.predict_proba(X)[:, 1]
    s1_pred = (s1_proba >= best_s1["threshold"]).astype(int)

    final_pred = np.array([NO_DEM] * len(X), dtype=object)
    dementia_idx = np.where(s1_pred == 1)[0]

    if len(dementia_idx) > 0:
        X_dem = X.iloc[dementia_idx] if hasattr(X, "iloc") else X[dementia_idx]
        s2_proba = stage2.predict_proba(X_dem)
        s2_pred_local = np.argmax(s2_proba, axis=1)
        s2_pred_local[s2_proba[:, vasc_local] >= best_s2_vasc["threshold"]] = vasc_local
        s2_pred_global = np.array([LOCAL_TO_GLOBAL[v] for v in s2_pred_local], dtype=object)
        final_pred[dementia_idx] = s2_pred_global

    return final_pred


test_pred_hier = hierarchical_predict(X_test)
hier_results = eval_predictions(
    y_test,
    test_pred_hier,
    title="Hierarchical CatBoost",
)
display(hier_results)


=== Hierarchical CatBoost ===
               precision    recall  f1-score   support

           AD     0.6909    0.7413    0.7152       603
NoDementia_Hy     0.9244    0.9157    0.9200      2217
       PD_LBD     0.6800    0.5862    0.6296        58
     Vascular     0.2667    0.1333    0.1778        30

     accuracy                         0.8649      2908
    macro avg     0.6405    0.5941    0.6107      2908
 weighted avg     0.8643    0.8649    0.8641      2908

Balanced accuracy: 0.5941213859775826
Macro F1: 0.6106541178840295
Weighted F1: 0.8640913160334263
[[ 447  145    6    5]
 [ 171 2030   10    6]
 [  15    9   34    0]
 [  14   12    0    4]]


,class,precision,recall,f1,support
0,AD,0.690881,0.741294,0.715200,603
1,NoDementia_Hy,0.924408,0.915652,0.920009,2217
2,PD_LBD,0.680000,0.586207,0.629630,58
3,Vascular,0.266667,0.133333,0.177778,30


In [18]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

def search_best_single_threshold(y_true, proba, target_class="Vascular", grid=None):
    """
    Finds the threshold that maximizes F1 for the specified class.
    """

    if grid is None:
        grid = np.linspace(0.01, 0.80, 80)

    # get class order from probability matrix
    classes = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]
    j = classes.index(target_class)

    best = None
    rows = []

    for thr in grid:

        pred = np.argmax(proba, axis=1)
        pred = np.array([classes[i] for i in pred])

        # override prediction when vascular probability exceeds threshold
        pred[proba[:, j] >= thr] = target_class

        p, r, f, s = precision_recall_fscore_support(
            y_true,
            pred,
            labels=[target_class],
            zero_division=0
        )

        row = {
            "threshold": thr,
            "precision": p[0],
            "recall": r[0],
            "f1": f[0],
            "support": s[0]
        }

        rows.append(row)

        if best is None or row["f1"] > best["f1"]:
            best = row

    return best

In [19]:
def predict_with_class_thresholds(proba, thresholds=None):
    classes = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

    pred = np.argmax(proba, axis=1)
    pred = np.array([classes[i] for i in pred])

    if thresholds is None:
        return pred

    for cls, thr in thresholds.items():
        j = classes.index(cls)
        pred[proba[:, j] >= thr] = cls

    return pred

In [20]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_clf = CalibratedClassifierCV(
    clf_best,
    method="isotonic",
    cv=3
)

calibrated_clf.fit(X_train, y_train)

test_proba_cal = calibrated_clf.predict_proba(X_test)

best_vasc_thr_cal = search_best_single_threshold(
    y_val,
    calibrated_clf.predict_proba(X_val),
    target_class="Vascular"
)

print("Best calibrated vascular threshold:", best_vasc_thr_cal)

test_pred_cal = predict_with_class_thresholds(
    test_proba_cal,
    thresholds={"Vascular": best_vasc_thr_cal["threshold"]}
)

eval_predictions(
    y_test,
    test_pred_cal,
    title="Calibrated Logistic + tuned vascular threshold"
)

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348:

Best calibrated vascular threshold: {'threshold': np.float64(0.2), 'precision': np.float64(0.4090909090909091), 'recall': np.float64(0.36), 'f1': np.float64(0.3829787234042553), 'support': np.int64(25)}
=== Calibrated Logistic + tuned vascular threshold ===
               precision    recall  f1-score   support

           AD     0.7324    0.5174    0.6064       603
NoDementia_Hy     0.8712    0.9522    0.9099      2217
       PD_LBD     0.6364    0.3621    0.4615        58
     Vascular     0.3462    0.3000    0.3214        30

     accuracy                         0.8435      2908
    macro avg     0.6465    0.5329    0.5748      2908
 weighted avg     0.8323    0.8435    0.8320      2908

Balanced accuracy: 0.5329173854492179
Macro F1: 0.5748237050598944
Weighted F1: 0.8319665081521314
[[ 312  273   10    8]
 [  96 2111    2    8]
 [  10   26   21    1]
 [   8   13    0    9]]


,class,precision,recall,f1,support
0,AD,0.732394,0.517413,0.606414,603
1,NoDementia_Hy,0.871234,0.952188,0.909914,2217
2,PD_LBD,0.636364,0.362069,0.461538,58
3,Vascular,0.346154,0.300000,0.321429,30


In [21]:
proba = clf_best.predict_proba(X_test)

VAS_IDX = list(clf_best.classes_).index("Vascular")

scaled_proba = proba.copy()
scaled_proba[:,VAS_IDX] *= 1.7

scaled_proba = scaled_proba / scaled_proba.sum(axis=1, keepdims=True)

best_thr = search_best_single_threshold(
    y_val,
    clf_best.predict_proba(X_val),
    target_class="Vascular"
)

test_pred_scaled = predict_with_class_thresholds(
    scaled_proba,
    thresholds={"Vascular": best_thr["threshold"]}
)

eval_predictions(
    y_test,
    test_pred_scaled,
    title="Logistic + vascular probability scaling"
)

=== Logistic + vascular probability scaling ===
               precision    recall  f1-score   support

           AD     0.6288    0.6882    0.6572       603
NoDementia_Hy     0.9253    0.8778    0.9009      2217
       PD_LBD     0.4868    0.6379    0.5522        58
     Vascular     0.1739    0.4000    0.2424        30

     accuracy                         0.8287      2908
    macro avg     0.5537    0.6510    0.5882      2908
 weighted avg     0.8474    0.8287    0.8366      2908

Balanced accuracy: 0.6509798289748278
Macro F1: 0.5881886133346321
Weighted F1: 0.8366320975394617
[[ 415  142   18   28]
 [ 223 1946   21   27]
 [  14    5   37    2]
 [   8   10    0   12]]


,class,precision,recall,f1,support
0,AD,0.628788,0.688226,0.657165,603
1,NoDementia_Hy,0.925345,0.877763,0.900926,2217
2,PD_LBD,0.486842,0.637931,0.552239,58
3,Vascular,0.173913,0.400000,0.242424,30


In [23]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, balanced_accuracy_score

# ---------- 1) Calibrate the logistic model ----------
calibrated_clf = CalibratedClassifierCV(
    estimator=clf_best,
    method="sigmoid",   # safer for small datasets than isotonic
    cv=3
)

calibrated_clf.fit(X_train, y_train)

val_proba_cal = calibrated_clf.predict_proba(X_val)
test_proba_cal = calibrated_clf.predict_proba(X_test)

classes = list(calibrated_clf.classes_)
VAS_IDX = classes.index("Vascular")

print("Model classes:", classes)

# ---------- 2) Function to scale vascular probability ----------
def apply_class_probability_scaling(proba, class_idx, scale):
    p = proba.copy()
    p[:, class_idx] *= scale
    p = p / p.sum(axis=1, keepdims=True)
    return p


# ---------- 3) Grid search for best scale + threshold ----------
scale_grid = [1.0, 1.1, 1.2, 1.3, 1.5, 1.7, 2.0]
thr_grid = np.linspace(0.05, 0.6, 56)

rows = []
best_combo = None

for scale in scale_grid:

    val_scaled = apply_class_probability_scaling(val_proba_cal, VAS_IDX, scale)

    best_thr = search_best_single_threshold(
        y_val,
        val_scaled,
        target_class="Vascular",
        grid=thr_grid
    )

    val_pred = predict_with_class_thresholds(
        val_scaled,
        thresholds={"Vascular": best_thr["threshold"]}
    )

    macro_f1 = f1_score(y_val, val_pred, average="macro")
    bal_acc = balanced_accuracy_score(y_val, val_pred)

    row = {
        "scale": scale,
        "threshold": best_thr["threshold"],
        "vascular_precision": best_thr["precision"],
        "vascular_recall": best_thr["recall"],
        "vascular_f1": best_thr["f1"],
        "macro_f1": macro_f1,
        "balanced_accuracy": bal_acc
    }

    rows.append(row)

    if best_combo is None or row["vascular_f1"] > best_combo["vascular_f1"]:
        best_combo = row

combo_table = pd.DataFrame(rows).sort_values(
    ["vascular_f1", "macro_f1"],
    ascending=False
)

print("Best combination from validation:")
print(best_combo)

display(combo_table)


# ---------- 4) Apply best setting to test ----------
test_scaled = apply_class_probability_scaling(
    test_proba_cal,
    VAS_IDX,
    best_combo["scale"]
)

test_pred_combo = predict_with_class_thresholds(
    test_scaled,
    thresholds={"Vascular": best_combo["threshold"]}
)

eval_predictions(
    y_test,
    test_pred_combo,
    title="Calibrated Logistic + vascular scaling + tuned threshold"
)

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348:

Model classes: ['AD', 'NoDementia_Hy', 'PD_LBD', 'Vascular']
Best combination from validation:
{'scale': 1.7, 'threshold': np.float64(0.16999999999999998), 'vascular_precision': np.float64(0.2682926829268293), 'vascular_recall': np.float64(0.44), 'vascular_f1': np.float64(0.3333333333333333), 'macro_f1': 0.5915802768451605, 'balanced_accuracy': 0.5661627552380054}


,scale,threshold,vascular_precision,vascular_recall,vascular_f1,macro_f1,balanced_accuracy
5,1.7,0.17,0.268293,0.44,0.333333,0.591580,0.566163
4,1.5,0.15,0.261905,0.44,0.328358,0.590264,0.566022
6,2.0,0.19,0.261905,0.44,0.328358,0.590264,0.566022
3,1.3,0.11,0.244898,0.48,0.324324,0.588496,0.574561
0,1.0,0.10,0.244444,0.44,0.314286,0.586403,0.565362
1,1.1,0.11,0.244444,0.44,0.314286,0.586403,0.565362
2,1.2,0.12,0.244444,0.44,0.314286,0.586403,0.565362


=== Calibrated Logistic + vascular scaling + tuned threshold ===
               precision    recall  f1-score   support

           AD     0.7731    0.4577    0.5750       603
NoDementia_Hy     0.8601    0.9648    0.9094      2217
       PD_LBD     0.6552    0.3276    0.4368        58
     Vascular     0.2571    0.3000    0.2769        30

     accuracy                         0.8401      2908
    macro avg     0.6364    0.5125    0.5495      2908
 weighted avg     0.8317    0.8401    0.8241      2908

Balanced accuracy: 0.5125287425965687
Macro F1: 0.5495358654071708
Weighted F1: 0.8241374796929672
[[ 276  307    7   13]
 [  63 2139    3   12]
 [  11   27   19    1]
 [   7   14    0    9]]


,class,precision,recall,f1,support
0,AD,0.773109,0.457711,0.575000,603
1,NoDementia_Hy,0.860072,0.964817,0.909439,2217
2,PD_LBD,0.655172,0.327586,0.436782,58
3,Vascular,0.257143,0.300000,0.276923,30
